# Content-based book recommendations

This notebook compares books using text features and cosine similarity. It demonstrates a full book-to-book similarity matrix followed by a separate title-query example. No user ratings or reading history are used, so this is a content-based approach rather than collaborative filtering.

## 1. Import libraries

Pandas handles book records, `CountVectorizer` converts text into term-count vectors, and cosine similarity compares those vectors.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Load the book metadata

The code expects `BX-Books.csv` in Google Drive, with columns for ISBN, title, author, publication year, publisher, and image URLs. Each row represents a book record. The CSV is read using semicolons and Latin-1 encoding; `on_bad_lines="skip"` skips malformed records rather than repairing them.

**Execution order:** in Google Colab, run the Drive-mount cell below before this loading cell. Outside Colab, replace the Drive path with the location of your CSV and omit the Colab mounting step.

In [2]:
import pandas as pd

df = pd.read_csv(
    '/content/drive/MyDrive/BX-Books.csv',
    on_bad_lines='skip',
    encoding='latin-1',
    sep=';'
)
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,classical mythology,markp.o.morford,2002,oxford university press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,clara callan,richardbrucewright,2001,harperflamingo canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,decision in normandy,carlod'este,1991,harperperennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,flu: the story of the great influenza pandemic...,ginabarikolata,1999,farrar straus giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,the mummies of urumchi,e.j.w.barber,1999,w. w. norton &amp; company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Inspect the loaded data

`df.info()` shows column names, data types, and non-null counts. Check that the expected columns were loaded and look for missing values before constructing text features.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270491 entries, 0 to 270490
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 270491 non-null  object
 1   Book-Title           270491 non-null  object
 2   Book-Author          270489 non-null  object
 3   Year-Of-Publication  270491 non-null  int64 
 4   Publisher            270489 non-null  object
 5   Image-URL-S          270491 non-null  object
 6   Image-URL-M          270491 non-null  object
 7   Image-URL-L          270491 non-null  object
dtypes: int64(1), object(7)
memory usage: 16.5+ MB


### Optional preprocessing examples

The following commented cells show alternative CSV loading, text cleaning, and export operations. They do not execute as written. The alternative loader assumes a file without a header, so check the actual file format before using it. The export example would overwrite the CSV at the specified path if enabled.

In [51]:
# import pandas as pd

# column_names = ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication',
#                 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']

# # Read CSV safely
# df = pd.read_csv(
#     '/content/drive/MyDrive/BX-Books.csv',
#     sep=';',
#     encoding='latin-1',
#     header=None,
#     names=column_names,
#     engine='python',
#     quotechar='"',
#     on_bad_lines='skip'  # skip problematic lines
# )



In [52]:
# import pandas as pd

# # Noms de colonnes souhaités
# column_names = ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication',
#                 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']

# # Lire le CSV
# df = pd.read_csv(
#     '/content/drive/MyDrive/BX-Books.csv',
#     sep=';',
#     encoding='latin-1',
#     header=None,
#     names=column_names,
#     engine='python',
#     quotechar='"',
#     on_bad_lines='skip'
# )


In [53]:
# # Exemple : minuscule et nettoyage des espaces
# df['Book-Title'] = df['Book-Title'].str.lower().str.strip()
# df['Book-Author'] = df['Book-Author'].str.lower().str.replace(' ', '')
# df['Publisher'] = df['Publisher'].str.lower().str.strip()


In [54]:
# df.to_csv('/content/drive/MyDrive/BX-Books.csv',
#           sep=';',           # même séparateur que l'original
#           encoding='latin-1',
#           index=False)       # ne pas écrire l'index Pandas


## 3. Remove duplicate titles

Preview the records and count repeated `Book-Title` values. `drop_duplicates` keeps the first record for each exact title, then the second count checks the result. Books with the same title may have different authors or editions; title-based deduplication removes those distinctions. This step runs before lowercasing, so capitalization variants may remain.

In [5]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,classical mythology,markp.o.morford,2002,oxford university press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,clara callan,richardbrucewright,2001,harperflamingo canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,decision in normandy,carlod'este,1991,harperperennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,flu: the story of the great influenza pandemic...,ginabarikolata,1999,farrar straus giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,the mummies of urumchi,e.j.w.barber,1999,w. w. norton &amp; company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270491 entries, 0 to 270490
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 270491 non-null  object
 1   Book-Title           270491 non-null  object
 2   Book-Author          270489 non-null  object
 3   Year-Of-Publication  270491 non-null  int64 
 4   Publisher            270489 non-null  object
 5   Image-URL-S          270491 non-null  object
 6   Image-URL-M          270491 non-null  object
 7   Image-URL-L          270491 non-null  object
dtypes: int64(1), object(7)
memory usage: 16.5+ MB


In [7]:
df.duplicated(subset='Book-Title').sum()

np.int64(32383)

In [8]:
df = df.drop_duplicates(subset='Book-Title')

In [9]:
df.duplicated(subset='Book-Title').sum()

np.int64(0)

## 4. Select a reproducible sample

The code samples 15,000 records without replacement, requiring at least that many records after deduplication. The fixed random state makes the selection reproducible for the same input data. Resetting the index aligns row positions with the similarity matrix created later.

Sampling limits the cost of pairwise comparison: 15,000 books produce 225 million similarity values, about 1.8 GB for a float64 matrix alone. A smaller sample can help when memory is limited.

In [10]:
sample_size = 15000
df = df.sample(n=sample_size, replace=False, random_state=490)

df = df.reset_index()
df = df.drop('index',axis=1)

In [11]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0515091936,one knee equals two feet and everything else y...,johnmadden,1987,jove books,http://images.amazon.com/images/P/0515091936.0...,http://images.amazon.com/images/P/0515091936.0...,http://images.amazon.com/images/P/0515091936.0...
1,0786880872,out of the fog: treatment options and coping s...,kevinmurphy,1995,hyperion,http://images.amazon.com/images/P/0786880872.0...,http://images.amazon.com/images/P/0786880872.0...,http://images.amazon.com/images/P/0786880872.0...
2,0140547126,there's a nightmare in my closet (pied piper b...,mercermayer,1992,puffin books,http://images.amazon.com/images/P/0140547126.0...,http://images.amazon.com/images/P/0140547126.0...,http://images.amazon.com/images/P/0140547126.0...
3,0486406504,lord jim (dover thrift editions),josephconrad,1999,dover publications,http://images.amazon.com/images/P/0486406504.0...,http://images.amazon.com/images/P/0486406504.0...,http://images.amazon.com/images/P/0486406504.0...
4,0916890686,abc of avalanche safety,e.r.lachapel,1978,mountaineers books,http://images.amazon.com/images/P/0916890686.0...,http://images.amazon.com/images/P/0916890686.0...,http://images.amazon.com/images/P/0916890686.0...


## 5. Normalize authors, titles, and publishers

`clean_text` lowercases author names and removes spaces, so a multiword name becomes a single token. The subsequent cells lowercase titles and publishers. These transformations reduce formatting differences; they do not resolve spelling variants or identify authors uniquely. Converting a missing author with `str` can also introduce the literal token `nan`.

In [12]:
def clean_text(author):
    result = str(author).lower()
    return(result.replace(' ',''))

In [13]:
df['Book-Author'] = df['Book-Author'].apply(clean_text)

In [14]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0515091936,one knee equals two feet and everything else y...,johnmadden,1987,jove books,http://images.amazon.com/images/P/0515091936.0...,http://images.amazon.com/images/P/0515091936.0...,http://images.amazon.com/images/P/0515091936.0...
1,0786880872,out of the fog: treatment options and coping s...,kevinmurphy,1995,hyperion,http://images.amazon.com/images/P/0786880872.0...,http://images.amazon.com/images/P/0786880872.0...,http://images.amazon.com/images/P/0786880872.0...
2,0140547126,there's a nightmare in my closet (pied piper b...,mercermayer,1992,puffin books,http://images.amazon.com/images/P/0140547126.0...,http://images.amazon.com/images/P/0140547126.0...,http://images.amazon.com/images/P/0140547126.0...
3,0486406504,lord jim (dover thrift editions),josephconrad,1999,dover publications,http://images.amazon.com/images/P/0486406504.0...,http://images.amazon.com/images/P/0486406504.0...,http://images.amazon.com/images/P/0486406504.0...
4,0916890686,abc of avalanche safety,e.r.lachapel,1978,mountaineers books,http://images.amazon.com/images/P/0916890686.0...,http://images.amazon.com/images/P/0916890686.0...,http://images.amazon.com/images/P/0916890686.0...


In [15]:
df['Book-Title'] = df['Book-Title'].str.lower()
df['Publisher'] = df['Publisher'].str.lower()

## 6. Construct text features

The code removes identifiers, image URLs, and publication year, then joins the remaining columns starting at position 1 into `data`. With the expected order of title, author, and publisher, this combines **author and publisher**, excluding the title. Inspect the printed text to confirm which fields contribute to similarity.

In [16]:
# combine all strings:
df2 = df.drop(['ISBN','Image-URL-S','Image-URL-M','Image-URL-L','Year-Of-Publication'],axis=1)

df2['data'] = df2[df2.columns[1:]].apply(
    lambda x: ' '.join(x.dropna().astype(str)),
    axis=1
)
print(df2['data'].head())

0              johnmadden jove books
1               kevinmurphy hyperion
2           mercermayer puffin books
3    josephconrad dover publications
4    e.r.lachapel mountaineers books
Name: data, dtype: object


### Convert metadata into counts

`CountVectorizer` creates a sparse matrix with one row per book and one column per vocabulary term. Each value counts a term in that book?s combined metadata. Shared tokens contribute to similarity; the representation does not understand a book?s plot or meaning.

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df2['data'])

## 7. Compare every pair of books

Cosine similarity compares vector directions. For these nonnegative count vectors, scores range from 0 (no shared terms) to 1 (identical directions). The result is a square matrix: entry `(i, j)` compares books at positions `i` and `j`. Nonzero vectors have self-similarity 1; empty vectors do not.

The full matrix is dense even though the input counts are sparse.

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(vectorized)

In [19]:
print(similarities)

[[1.         0.         0.33333333 ... 0.         0.         0.        ]
 [0.         1.         0.         ... 0.         0.         0.        ]
 [0.33333333 0.         1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]


### Label the similarity matrix

Titles become row and column labels, making the scores easier to inspect. This assignment replaces `df` with the similarity table, so it no longer holds the original metadata. This section builds the table but does not yet select recommendations. For book-to-book recommendations, exclude the selected book itself when ranking scores.

In [20]:
df = pd.DataFrame(similarities, columns=df['Book-Title'], index=df['Book-Title']).reset_index()

df.head()

Book-Title,Book-Title,one knee equals two feet and everything else you need to know about football,out of the fog: treatment options and coping strategies for adult attention deficit disorders,there's a nightmare in my closet (pied piper book),lord jim (dover thrift editions),abc of avalanche safety,after the crash: new investment strategies for survivors from america's foremost financial experts,living alone and liking it,"a cowboy's pursuit (man of the month / 50th book / code of the west) (silhoutte desire, no. 1441)",the winter's tale (folger library general reader's shakespeare),...,john adams,the marrying macallister : the baby bet: macallisters gifts (special edition),winter moon,schmatz. oder die sackgasse.,football's women: the forgotten heroes,his unexpected wife,dillinger: a novel,amerikanische reise: roman,livres du rail,healing the wounds: the promise of ecofeminism
0,one knee equals two feet and everything else y...,1.000000,0.0,0.333333,0.0,0.333333,0.333333,0.333333,0.0,0.333333,...,0.0,0.0,0.288675,0.0,0.333333,0.0,0.0,0.0,0.0,0.0
1,out of the fog: treatment options and coping s...,0.000000,1.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,there's a nightmare in my closet (pied piper b...,0.333333,0.0,1.000000,0.0,0.333333,0.333333,0.333333,0.0,0.333333,...,0.0,0.0,0.288675,0.0,0.333333,0.0,0.0,0.0,0.0,0.0
3,lord jim (dover thrift editions),0.000000,0.0,0.000000,1.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
4,abc of avalanche safety,0.333333,0.0,0.333333,0.0,1.000000,0.333333,0.333333,0.0,0.333333,...,0.0,0.0,0.288675,0.0,0.333333,0.0,0.0,0.0,0.0,0.0


In [21]:
print(df.columns)   # lst the column names


Index(['Book-Title',
       'one knee equals two feet and everything else you need to know about football',
       'out of the fog: treatment options and coping strategies for adult attention deficit disorders',
       'there's a nightmare in my closet (pied piper book)',
       'lord jim (dover thrift editions)', 'abc of avalanche safety',
       'after the crash: new investment strategies for survivors from america's foremost financial experts',
       'living alone and liking it',
       'a cowboy's pursuit  (man of the month / 50th book / code of the west) (silhoutte desire, no. 1441)',
       'the winter's tale (folger library general reader's shakespeare)',
       ...
       'john adams',
       'the marrying macallister : the baby bet:  macallisters gifts (special edition)',
       'winter moon', 'schmatz. oder die sackgasse.',
       'football's women: the forgotten heroes', 'his unexpected wife',
       'dillinger: a novel', 'amerikanische reise: roman', 'livres du rail',
    

## 8. Retrieve books from a title query

This self-contained example reloads the CSV and rebuilds text features from **title, author, and publisher**. It does not reuse the earlier sample or deduplication. Its `header=None` setting assumes a headerless CSV; if the file has a header, that row will be treated as data.

The input title is transformed using the fitted vocabulary and compared with every book. Only one query-to-catalog score vector is calculated, avoiding the full pairwise matrix. Scores are sorted in descending order and the ten highest are displayed.

This is text matching against a typed title, rather than a lookup of the selected book?s complete metadata. An exact match can appear in the results, duplicates are possible, and a query containing only unknown terms produces zero scores.

### Interpret the results

A high score indicates overlapping text features, not a probability that someone will enjoy the book. Inspect whether the results are useful and varied. No held-out relevance judgments or user feedback are evaluated in this notebook.

In [36]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# --- CSV preprocessing (as before) ---
column_names = ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication',
                'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']

df = pd.read_csv('/content/drive/MyDrive/BX-Books.csv', sep=';', encoding='latin-1',
                 header=None, names=column_names, engine='python', quotechar='"', on_bad_lines='skip')

# Cleaning
df['Book-Title'] = df['Book-Title'].str.lower().str.strip()
df['Book-Author'] = df['Book-Author'].str.lower().str.replace(' ', '')
df['Publisher'] = df['Publisher'].str.lower().str.strip()

# Combine columns
df['data'] = df[['Book-Title', 'Book-Author', 'Publisher']].apply(lambda x: ' '.join(x.dropna()), axis=1)

# Vectorization
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df['data'])

# Input book
input_book = 'far beyond the stars (star trek deep space nine)'
input_processed = input_book.lower().strip()

# Vectorize the input
input_vector = vectorizer.transform([input_processed])

# Compute cosine similarity
similarities = cosine_similarity(input_vector, vectorized).flatten()

# Top 10 books with similarity score
top_indices = similarities.argsort()[::-1][0:10]
top_books = df.iloc[top_indices][['Book-Title']].copy()
top_books['similarity_score'] = similarities[top_indices]

print(top_books)


                                               Book-Title  similarity_score
188863   far beyond the stars (star trek deep space nine)          0.889297
154747  the deep space nine technical manual (star tre...          0.748455
23449              the search (star trek deep space nine)          0.712697
116657  star trek deep space nine warped (star trek de...          0.695048
239958  deep space nine companion (star trek deep spac...          0.695048
36171         the siege (star trek deep space nine, no 2)          0.688530
62777   the way of the warrior (star trek deep space n...          0.688247
95665   worlds of deep space nine, vol i (star trek de...          0.680414
123258        the star ghost (star trek: deep space nine)          0.673575
95777   the mist:  the captain's table book 3 (star tr...          0.670820
